# DocSmile — Docling single-book test (text-only Markdown)

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (e.g. **T4**).

This matches `PIPELINE_SUMMARY.md` §5.3: OCR on, table structure on, **no** picture extraction / VLM; Markdown export **without** figure noise (`strict_text=True`).


In [ ]:
# Install Docling (pulls compatible PyTorch for Colab)
%pip install -q docling

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
# Upload ONE PDF (your dental book)
# Alternative: `!wget -q 'https://arxiv.org/pdf/2408.09869.pdf' -O sample.pdf` then `PDF_PATH = Path('sample.pdf')`
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # pick a single .pdf
assert uploaded, "Upload one PDF (or comment this out and set PDF_PATH manually)."
PDF_PATH = Path(next(iter(uploaded.keys())))
print("Input:", PDF_PATH.resolve(), "size:", PDF_PATH.stat().st_size, "bytes")

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TableFormerMode,
    EasyOcrOptions,
)
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

# --- PIPELINE_SUMMARY.md §5.3 ---
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True

# No embedded figure extraction for this CPT pass
pipeline_options.generate_picture_images = False

# Page renders are still used internally when OCR runs on scanned pages
pipeline_options.generate_page_images = True

# Optional: disable formula VLM path (text CPT only)
if hasattr(pipeline_options, "do_formula_enrichment"):
    pipeline_options.do_formula_enrichment = False
if hasattr(pipeline_options, "do_picture_description"):
    pipeline_options.do_picture_description = False

# GPU/CPU: set here (EasyOCR follows this — do not use deprecated EasyOcrOptions.use_gpu)
_accel = AcceleratorDevice.CUDA if torch.cuda.is_available() else AcceleratorDevice.CPU
pipeline_options.accelerator_options = AcceleratorOptions(device=_accel)

pipeline_options.ocr_options = EasyOcrOptions(
    lang=["en"],
    confidence_threshold=0.5,
)

pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
    mode=TableFormerMode.ACCURATE,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

print("Accelerator:", _accel)
print("Converting… (large books can take several minutes)")
result = converter.convert(str(PDF_PATH))
doc = result.document
print("Done.")

In [ ]:
# Text-oriented Markdown: suppresses rich image markup (good for CPT corpus)
md_strict = doc.export_to_markdown(strict_text=True)

# Also keep default MD for side-by-side comparison (optional)
md_default = doc.export_to_markdown(strict_text=False)

OUT = Path("book_text_only.md")
OUT.write_text(md_strict, encoding="utf-8")
print("Saved:", OUT.resolve(), "chars:", len(md_strict))
print("\n--- First 2500 chars (strict_text) ---\n")
print(md_strict[:2500])

In [ ]:
# Download the Markdown file to your machine
from google.colab import files as colab_files
colab_files.download(str(OUT))

## Notes

- **`strict_text=True`:** Uses Docling’s **text-focused** Markdown export (good default for CPT; avoids embedding image markup in the string).
- If your PDF is **native text** (not scanned), you can try `do_ocr = False` for speed.
- If you hit **OOM** on T4, lower `images_scale` (e.g. `1.0`), shorten the test PDF, or use `ThreadedPdfPipelineOptions` with smaller batch sizes.
- Docling versions differ slightly; if an attribute is missing, remove that line or upgrade: `pip install -U docling`.
- **First run:** EasyOCR downloads **detection + recognition** weights (can take several minutes). Later runs use the cache.
- **`HF_TOKEN` warning:** Harmless if you only use public models. Add a [Hugging Face token](https://huggingface.co/settings/tokens) under Colab **Secrets** only if you need Hub authentication.
